# Stage B — trọng số nhóm NORMAL khó

Khởi tạo từ năm checkpoint DenseNet121 của v5, **không** train lại từ ImageNet.
Thay đổi phương pháp duy nhất là trọng số tương đối của những ảnh NORMAL mà hai
mô hình đã đóng băng đều thấy khó.

---

## Mốc cần vượt

| | Độ đặc hiệu | FP | Độ nhạy |
|---|---:|---:|---:|
| DenseNet121 v5 | 0,8044 | 44 | 0,9951 |
| Ensemble ResNet+DenseNet | 0,8222 | 40 | 0,9951 |

Mục tiêu Stage B: **≥0,82 hoặc tăng ≥0,02 so với DenseNet**, giảm ròng ít nhất
4 ca báo nhầm, độ nhạy giữ ≥0,97.

## Cần gắn vào notebook trên Kaggle

```
Add Data  → paultimothymooney/chest-xray-pneumonia
Add Input → Your Work → Notebook → bản v5 chạy THÀNH CÔNG
Add Input → Your Work → Notebook → bản v4 (cho OOF của ResNet)
```

Notebook tự tìm checkpoint theo tên file, không phụ thuộc slug của mount. Nếu
gắn nhầm hai version của v5 thì nó dừng ngay thay vì chọn bừa.

## Bốn thứ được kiểm trước khi có bất kỳ bước tối ưu nào

1. đúng năm checkpoint, mỗi fold một cái, `strict=True`;
2. cấu hình nguồn đúng là DenseNet121 + stretch + augment mạnh + weighted;
3. bảng hardness dựng lại từ OOF khớp băm đã khóa ở commit `b3036dd`;
4. **epoch 0 tái hiện đúng dự đoán v5 trên cả năm fold** — lệch ≤ 1e-5, tương
   quan ≥ 0,999999.

Điểm 4 quan trọng nhất: nếu trượt thì trọng số đang load không phải trọng số đã
sinh ra kết quả v5, và mọi so sánh phía sau đều vô nghĩa.

> Epoch 0 là ứng viên hợp lệ trong việc chọn checkpoint. Fold nào fine-tune làm
> tệ đi thì giữ nguyên checkpoint cũ.

## Cấu hình

In [1]:
RUN_MODE      = "auto"   # auto | smoke | full
DATA_ROOT_OVERRIDE = None # ví dụ: "/Users/me/data/chest_xray"
SEED          = 42
IMG_SIZE      = 224
BATCH_SIZE    = 32
EPOCHS        = 12
LR            = 1e-4
WEIGHT_DECAY  = 1e-5
PATIENCE      = 5         # phải lớn hơn scheduler patience để LR giảm còn có epoch phát huy
SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR   = 0.3
MIN_LR             = 1e-6
NUM_WORKERS   = 2         # runtime sẽ ép về 0 trên macOS

N_FOLDS       = 5         # 1 = một holdout 15%; 5 = cross-validation đầy đủ
VAL_FRACTION  = 0.15      # chỉ dùng khi N_FOLDS = 1
BORDER_FRAC   = 0.15      # dùng ở phần 4.3
DETERMINISTIC = True      # cudnn tất định; chậm hơn một chút, đổi lại tái lập tốt hơn
RESIZE_MODE   = "letterbox"  # mặc định cho thí nghiệm không ghi rõ "resize"
THRESHOLD_OBJECTIVE = "sensitivity"  # sensitivity | balanced_accuracy
CHECKPOINT_TIE_MARGIN = 0.005  # chênh độ đặc hiệu dưới mức này coi như bằng nhau
TARGET_SENSITIVITY = 0.97
BOOTSTRAP_REPS = 2000    # KTC cho audit tỉ lệ khung ở mức filename-derived group

# Hai bộ augmentation. "mạnh" mô phỏng thiết lập của các cài đặt công khai
# đạt độ đặc hiệu cao hơn: xoay 30 độ, zoom và dịch ảnh.
AUG_PRESETS = {
    "nhe":  {"rotation": 10, "scale": 0.00, "translate": 0.00, "jitter": 0.15},
    "manh": {"rotation": 30, "scale": 0.20, "translate": 0.10, "jitter": 0.20},
}

# Stage B khởi tạo từ checkpoint DenseNet121 của v5, không train từ ImageNet.
# Thay đổi phương pháp DUY NHẤT là trọng số tương đối của nhóm NORMAL khó.
HARD_MULTIPLIER   = 2.0    # nhân vào loss của group NORMAL khó
HARD_FRACTION     = 0.25   # tỉ lệ NORMAL trong training split bị đánh dấu
FINETUNE_EPOCHS   = 3      # cộng thêm epoch 0 = checkpoint gốc
FINETUNE_LR       = 1e-5
FINETUNE_DECAY    = 1e-5

STAGE_B_EXPERIMENT = {
    "name": "densenet_hn2", "arch": "densenet121", "size": 224,
    "aug": "manh", "balancing": "weighted", "resize": "stretch",
    "hoi": "DenseNet121 + trọng số nhóm NORMAL khó",
}
EXPERIMENTS = [STAGE_B_EXPERIMENT]

# Cấu hình nguồn phải khớp, nếu không thì checkpoint không dùng lại được.
EXPECTED_SOURCE = {"arch": "densenet121", "size": 224, "resize": "stretch",
                   "aug": "manh", "balancing": "weighted"}
SOURCE_EXPERIMENT = "densenet121_robust"

# Băm của bảng hard-negative đã khóa ở commit b3036dd. Notebook dựng lại bảng
# từ OOF rồi đối chiếu, nên một thay đổi im lặng ở phía nguồn sẽ lộ ra.
EXPECTED_HARDNESS_HASHES = {
    "fold0": "b6c6a6a3d8891724", "fold1": "25081fc3411338f3",
    "fold2": "068bea3a87521f51", "fold3": "5bcf6bf8fb177992",
    "fold4": "2c7bbc70c79d15c0",
}

# Epoch 0 phải tái hiện đúng dự đoán đã lưu của v5, nếu không thì checkpoint
# đang load không phải checkpoint đã sinh ra kết quả v5.
EPOCH0_MAX_ABS_DIFF = 1e-5
EPOCH0_MIN_CORRELATION = 0.999999

BASELINE_DENSENET = {"auc": 0.9801, "sensitivity": 0.9951,
                     "specificity": 0.8044, "tn": 181, "fp": 44,
                     "fn": 1, "tp": 202}
BASELINE_ENSEMBLE = {"specificity": 0.8222, "fp": 40}

CLASSES = ("NORMAL", "PNEUMONIA")  # NORMAL=0, PNEUMONIA=1

EXPECTED_STAGE_A1 = {
    "arch": "densenet121", "size": 224, "resize": "stretch",
    "aug": "manh", "balancing": "weighted",
}

In [2]:
import gc, hashlib, json, os, platform, random, re, time, warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import scipy
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

warnings.filterwarnings("ignore", category=UserWarning)

IS_KAGGLE = Path("/kaggle/working").is_dir()
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

if RUN_MODE == "auto":
    RUN_MODE = "full" if IS_KAGGLE and DEVICE.type == "cuda" else "smoke"
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE phải là 'auto', 'smoke' hoặc 'full'")

if RUN_MODE == "smoke":
    # N_FOLDS giữ nguyên 5: đổi nó sẽ đổi luôn cách chia, và dự đoán
    # epoch 0 sẽ không còn so được với bản v5 đã lưu.
    FINETUNE_EPOCHS, FOLDS_TO_RUN = 1, [0]
    if DEVICE.type != "cuda":
        BATCH_SIZE = min(BATCH_SIZE, 16)
else:
    FOLDS_TO_RUN = list(range(N_FOLDS))

if platform.system() == "Darwin":
    NUM_WORKERS = 0  # notebook + spawn không an toàn với closure worker/cache global
LOCAL_PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                           if (p / ".git").is_dir()), Path.cwd())
WORK_DIR = (Path("/kaggle/working") if IS_KAGGLE
            else LOCAL_PROJECT_ROOT / "artifacts/notebook_rerun")
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = WORK_DIR / "train_log_stage_b.txt"
LOG_PATH.write_text("", encoding="utf-8")
PIN_MEMORY = DEVICE.type == "cuda"
AMP_ENABLED = DEVICE.type == "cuda"
DEVICE_NAME = (torch.cuda.get_device_name(0) if DEVICE.type == "cuda"
               else "Apple Metal (MPS)" if DEVICE.type == "mps" else platform.processor() or "CPU")

if DETERMINISTIC and DEVICE.type == "cuda":
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def set_seed(seed=SEED):
    """Seed mọi nguồn ngẫu nhiên mà pipeline đụng tới."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def loader_seed_args(seed=SEED):
    """generator + worker_init_fn cho DataLoader.

    Thiếu hai thứ này thì thứ tự xáo trộn và augmentation chạy trong worker vẫn
    ngẫu nhiên dù đã gọi set_seed — một lỗ hổng tái lập rất hay bị bỏ sót.
    """
    generator = torch.Generator()
    generator.manual_seed(seed)

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        random.seed(worker_seed)
        np.random.seed(worker_seed)

    return {"generator": generator, "worker_init_fn": worker_init_fn}


def log(*parts):
    """In ra màn hình, đồng thời ghi vào LOG_PATH.

    Output của notebook Kaggle có thể mất chunk khi in nhanh; file thì không.
    """
    line = " ".join(str(part) for part in parts)
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")


set_seed()

# Ghi lại phiên bản thư viện. Thiếu nó thì con số trong báo cáo không gắn được
# với môi trường đã sinh ra chúng.
VERSIONS = {
    "python": platform.python_version(), "torch": torch.__version__,
    "torchvision": torchvision.__version__, "numpy": np.__version__,
    "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "pillow": PIL.__version__,
}
with open(WORK_DIR / "environment.json", "w") as handle:
    json.dump({**VERSIONS, "device": str(DEVICE), "device_name": DEVICE_NAME,
               "run_mode": RUN_MODE, "seed": SEED,
               "deterministic": DETERMINISTIC}, handle, indent=2)

# Lưu cấu hình đã resolve sau khi auto/smoke/full được áp dụng. File này giúp
# phân biệt source config với config thực sự sinh ra kết quả.
RESOLVED_CONFIG = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "image_cache_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "finetune_epochs": FINETUNE_EPOCHS,
    "hard_multiplier": HARD_MULTIPLIER,
    "hard_fraction": HARD_FRACTION,
    "learning_rate": FINETUNE_LR,
    "weight_decay": WEIGHT_DECAY,
    "patience": PATIENCE,
    "scheduler_patience": SCHEDULER_PATIENCE,
    "scheduler_factor": SCHEDULER_FACTOR,
    "min_lr": MIN_LR,
    "checkpoint_tie_margin": CHECKPOINT_TIE_MARGIN,
    "n_folds": N_FOLDS,
    "val_fraction": VAL_FRACTION,
    "deterministic": DETERMINISTIC,
    "threshold_objective": THRESHOLD_OBJECTIVE,
    "target_sensitivity": TARGET_SENSITIVITY,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "augment_presets": AUG_PRESETS,
    "experiments": EXPERIMENTS,
}
with open(WORK_DIR / "resolved_config.json", "w", encoding="utf-8") as handle:
    json.dump(RESOLVED_CONFIG, handle, indent=2, ensure_ascii=False)

log("runtime:", "Kaggle" if IS_KAGGLE else "local", "| mode:", RUN_MODE)
log("device:", DEVICE, f"({DEVICE_NAME})", "| AMP:", AMP_ENABLED,
    "| workers:", NUM_WORKERS)
if RUN_MODE == "smoke":
    log("SMOKE RUN: chỉ kiểm tra pipeline; KHÔNG dùng chỉ số để báo cáo.")
log("phiên bản:", " ".join(f"{k}={v}" for k, v in VERSIONS.items()))

runtime: Kaggle | mode: full
device: cuda (Tesla T4) | AMP: True | workers: 2
phiên bản: python=3.12.13 torch=2.10.0+cu128 torchvision=0.25.0+cu128 numpy=2.0.2 pandas=2.3.3 scikit-learn=1.6.1 scipy=1.16.3 pillow=11.3.0


# 1. Dữ liệu

Giữ nguyên từ v5: cùng manifest, cùng group, cùng fold.

In [3]:
def list_images(directory):
    """Ảnh .jpeg thật, bỏ file sidecar ._* của macOS."""
    return sorted(p for p in Path(directory).glob("*.jpeg")
                  if not p.name.startswith("._"))


def find_data_root(search_paths):
    """Thư mục chứa trực tiếp train/NORMAL và train/PNEUMONIA."""
    if isinstance(search_paths, (str, Path)):
        search_paths = [search_paths]

    candidates = []
    for base in map(Path, search_paths):
        if not base.exists():
            continue
        for train_dir in base.rglob("train"):
            if "__MACOSX" in train_dir.parts:
                continue
            if (train_dir / "NORMAL").is_dir() and (train_dir / "PNEUMONIA").is_dir():
                candidates.append(train_dir.parent.resolve())

    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy dataset dưới {[str(p) for p in search_paths]}. "
            "Kaggle: Add Data 'Chest X-Ray Images (Pneumonia)'. "
            "Mac: đặt DATA_ROOT_OVERRIDE hoặc CXR_DATA_ROOT.")

    candidates = sorted(set(candidates), key=lambda path: len(path.parts))
    for candidate in candidates:
        n = len(list_images(candidate / "train" / "NORMAL"))
        mark = "  <- dùng" if candidate == candidates[0] else "  (bản trùng, bỏ qua)"
        print(f"  {candidate}  [{n} ảnh train/NORMAL]{mark}")
    return candidates[0]


explicit_root = DATA_ROOT_OVERRIDE or os.environ.get("CXR_DATA_ROOT")
if explicit_root:
    DATA_ROOT = find_data_root([explicit_root])
else:
    cwd = Path.cwd()
    DATA_ROOT = find_data_root([
        "/kaggle/input", cwd / "chest_xray", cwd.parent / "chest_xray",
        cwd.parent.parent / "chest_xray", cwd / "data/raw",
        cwd.parent / "data/raw",
    ])
log("\nDATA_ROOT =", DATA_ROOT)

  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray  [1341 ảnh train/NORMAL]  <- dùng
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray  [1341 ảnh train/NORMAL]  (bản trùng, bỏ qua)

DATA_ROOT = /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray


In [4]:
PNEUMONIA_RE = re.compile(r"^person(\d+)_(bacteria|virus)_", re.IGNORECASE)
NORMAL_RE    = re.compile(r"^(?:(NORMAL\d+)-)?IM-(\d+)-", re.IGNORECASE)


def parse_group_id(filename):
    """Khoá group suy từ tên file; không khẳng định đây là clinical patient ID."""
    match = PNEUMONIA_RE.match(filename)
    if match:
        return f"pneumonia:{match.group(2).lower()}:{int(match.group(1))}"
    match = NORMAL_RE.match(filename)
    if match:
        return f"normal:{(match.group(1) or 'IM').lower()}:{int(match.group(2))}"
    raise ValueError(f"Tên file lạ, không suy ra được group: {filename}")


def build_manifest(root):
    """Một dòng cho mỗi ảnh: đường dẫn, split gốc, nhãn, filename-derived group."""
    rows = []
    for split in ("train", "val", "test"):
        for class_id, class_name in enumerate(CLASSES):
            directory = Path(root) / split / class_name
            if not directory.is_dir():
                continue
            for path in list_images(directory):
                rows.append({
                    "path": str(path.resolve()), "filename": path.name,
                    "split_original": split, "class_name": class_name,
                    "class_id": class_id, "group_id": parse_group_id(path.name)})
    if not rows:
        raise FileNotFoundError(f"Không có ảnh .jpeg nào dưới {root}")

    frame = pd.DataFrame(rows)
    frame["cache_index"] = np.arange(len(frame))   # vị trí trong cache ở mục 2.2
    return frame


manifest = build_manifest(DATA_ROOT)
log(f"{len(manifest):,} ảnh | {manifest['group_id'].nunique():,} filename-derived groups")
manifest.head(3)

5,856 ảnh | 4,097 filename-derived groups


,path,filename,split_original,class_name,class_id,group_id,cache_index
0,/kaggle/input/datasets/paultimothymooney/chest...,IM-0115-0001.jpeg,train,NORMAL,0,normal:im:115,0
1,/kaggle/input/datasets/paultimothymooney/chest...,IM-0117-0001.jpeg,train,NORMAL,0,normal:im:117,1
2,/kaggle/input/datasets/paultimothymooney/chest...,IM-0119-0001.jpeg,train,NORMAL,0,normal:im:119,2


In [5]:
print("1.3.1  Số lượng theo split và lớp")
print("-" * 62)
print(f"{'split':<8}{'NORMAL':>9}{'PNEUMONIA':>12}{'tổng':>9}{'P/N':>7}")
for split in ("train", "val", "test"):
    subset = manifest[manifest["split_original"] == split]
    counts = subset["class_name"].value_counts()
    normal, pneumonia = int(counts.get("NORMAL", 0)), int(counts.get("PNEUMONIA", 0))
    ratio = pneumonia / normal if normal else float("nan")
    print(f"{split:<8}{normal:>9,}{pneumonia:>12,}{normal + pneumonia:>9,}{ratio:>7.2f}")
print(f"{'TỔNG':<8}{'':>9}{'':>12}{len(manifest):>9,}")

1.3.1  Số lượng theo split và lớp
--------------------------------------------------------------
split      NORMAL   PNEUMONIA     tổng    P/N
train       1,341       3,875    5,216   2.89
val             8           8       16   1.00
test          234         390      624   1.67
TỔNG                             5,856


In [6]:
print("1.3.2  Ảnh trùng nội dung (SHA-256)")
print("-" * 62)
manifest["sha256"] = [hashlib.sha256(Path(path).read_bytes()).hexdigest()
                      for path in manifest["path"]]
by_hash = defaultdict(list)
for digest, split in zip(manifest["sha256"], manifest["split_original"]):
    by_hash[digest].append(split)

duplicates = [s for s in by_hash.values() if len(s) > 1]
cross_split = [s for s in duplicates if len(set(s)) > 1]
print(f"tổng file             : {len(manifest):,}")
print(f"hash duy nhất         : {len(by_hash):,}")
print(f"nhóm ảnh trùng        : {len(duplicates)}")
print(f"  trong đó xuyên split: {len(cross_split)}")
print()
print("Không có ảnh y hệt nằm xuyên original split." if not cross_split
      else "CẢNH BÁO: ảnh trùng xuyên original split — kết quả đánh giá bị nhiễm.")

1.3.2  Ảnh trùng nội dung (SHA-256)
--------------------------------------------------------------
tổng file             : 5,856
hash duy nhất         : 5,824
nhóm ảnh trùng        : 30
  trong đó xuyên split: 0

Không có ảnh y hệt nằm xuyên original split.


In [7]:
print("1.3.4  Filename-derived group và nguy cơ trùng giữa các split")
print("-" * 62)
naive_re = re.compile(r"^(person\d+)_", re.IGNORECASE)
naive, corrected = defaultdict(set), defaultdict(set)
for filename, split in zip(manifest["filename"], manifest["split_original"]):
    match = naive_re.match(filename)
    naive[match.group(1).lower() if match else filename].add(split)
    corrected[parse_group_id(filename)].add(split)

naive_span     = sum(1 for s in naive.values() if len(s) > 1)
corrected_span = sum(1 for s in corrected.values() if len(s) > 1)
print(f"khoá person<N>              : {len(naive):,} nhóm, {naive_span} nằm ở >1 split")
print(f"khoá (phân nhóm, person<N>) : {len(corrected):,} nhóm, {corrected_span} nằm ở >1 split")

subtype_ids = defaultdict(set)
for filename in manifest["filename"]:
    match = PNEUMONIA_RE.match(filename)
    if match:
        subtype_ids[match.group(2).lower()].add(int(match.group(1)))
print()
for subtype, ids in sorted(subtype_ids.items()):
    print(f"  {subtype:<9}: {len(ids):,} số, dải 1..{max(ids)}, "
          f"mật độ {len(ids) / max(ids):.3f}")
print(f"  số được dùng bởi CẢ HAI phân nhóm: "
      f"{len(subtype_ids['bacteria'] & subtype_ids['virus']):,}")

per_group = Counter(parse_group_id(f) for f in manifest["filename"])
multi = sum(1 for n in per_group.values() if n > 1)
print(f"\nnhóm có >1 ảnh: {multi:,}/{len(per_group):,} "
      f"(nhiều nhất {max(per_group.values())} ảnh)")

1.3.4  Filename-derived group và nguy cơ trùng giữa các split
--------------------------------------------------------------
khoá person<N>              : 3,257 nhóm, 170 nằm ở >1 split
khoá (phân nhóm, person<N>) : 4,097 nhóm, 0 nằm ở >1 split

  bacteria : 1,437 số, dải 1..1954, mật độ 0.735
  virus    : 1,216 số, dải 1..1685, mật độ 0.722
  số được dùng bởi CẢ HAI phân nhóm: 979

nhóm có >1 ảnh: 726/4,097 (nhiều nhất 30 ảnh)


# 2. Phương pháp

In [8]:
def label_split(manifest, train, val, test):
    out = manifest.copy()
    out["split"] = pd.NA
    out.loc[train.index, "split"] = "train"
    out.loc[val.index,   "split"] = "val"
    out.loc[test.index,  "split"] = "test"
    return out


def make_folds(manifest, n_folds=N_FOLDS, val_fraction=VAL_FRACTION, seed=SEED):
    """Danh sách manifest, mỗi phần tử là một fold đã gán cột split."""
    pool = manifest[manifest["split_original"].isin(["train", "val"])]
    test = manifest[manifest["split_original"] == "test"]
    n_splits = max(2, round(1 / val_fraction)) if n_folds == 1 else n_folds
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    folds = [label_split(manifest, pool.iloc[train_idx], pool.iloc[val_idx], test)
             for train_idx, val_idx in
             splitter.split(pool, pool["class_id"], groups=pool["group_id"])]
    return folds[:1] if n_folds == 1 else folds


def count_leaked_groups(split):
    """Số filename-derived groups xuất hiện ở nhiều split. Phải bằng 0."""
    return int((split.groupby("group_id")["split"].nunique() > 1).sum())


def count_leaked_hashes(split):
    """Số nội dung ảnh y hệt xuất hiện ở nhiều split. Phải bằng 0."""
    return int((split.groupby("sha256")["split"].nunique() > 1).sum())


def split_summary(split):
    rows = []
    for name in ("train", "val", "test"):
        subset = split[split["split"] == name]
        counts = subset["class_name"].value_counts()
        normal, pneumonia = int(counts.get("NORMAL", 0)), int(counts.get("PNEUMONIA", 0))
        rows.append({"split": name, "NORMAL": normal, "PNEUMONIA": pneumonia,
                     "tổng": normal + pneumonia,
                     "groups": subset["group_id"].nunique(),
                     "P/N": round(pneumonia / max(normal, 1), 2)})
    return pd.DataFrame(rows).set_index("split")


FOLDS = make_folds(manifest)
log(f"\n{len(FOLDS)} fold, chia theo filename-derived group:")
for i, split in enumerate(FOLDS):
    s = split_summary(split)
    log(f"  fold {i}: train {s.loc['train','tổng']:>5,}  val {s.loc['val','tổng']:>4,}  "
        f"test {s.loc['test','tổng']:>4,}  |  group/hash ở >1 split: "
        f"{count_leaked_groups(split)}/{count_leaked_hashes(split)}")
    assert count_leaked_groups(split) == 0
    assert count_leaked_hashes(split) == 0
    split.to_csv(WORK_DIR / f"manifest_fold{i}.csv", index=False)

print("\nChi tiết fold 0:")
display(split_summary(FOLDS[0]))


5 fold, chia theo filename-derived group:
  fold 0: train 4,168  val 1,064  test  624  |  group/hash ở >1 split: 0/0
  fold 1: train 4,224  val 1,008  test  624  |  group/hash ở >1 split: 0/0
  fold 2: train 4,165  val 1,067  test  624  |  group/hash ở >1 split: 0/0
  fold 3: train 4,190  val 1,042  test  624  |  group/hash ở >1 split: 0/0
  fold 4: train 4,181  val 1,051  test  624  |  group/hash ở >1 split: 0/0

Chi tiết fold 0:


,NORMAL,PNEUMONIA,tổng,groups,P/N
split,,,,,
train,1092,3076,4168,2934,2.82
val,257,807,1064,735,3.14
test,234,390,624,428,1.67


## 2.2. Tiền xử lý và augmentation

In [9]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
MEAN_T = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_T  = torch.tensor(IMAGENET_STD,  device=DEVICE).view(1, 3, 1, 1)


def device_augment(batch, cfg):
    """Lật, xoay, zoom, dịch, đổi sáng/tương phản trên CUDA/MPS/CPU.

    Làm bằng PIL trong DataLoader thì CPU thành nút thắt: riêng RandomRotation
    và ColorJitter đã ngốn hơn 1.000 lần thời gian đọc cache. Ở đây mọi phép
    biến đổi là tensor op chạy theo lô trên accelerator đang chọn.

    Xoay, zoom và dịch được gộp vào MỘT phép biến đổi affine, nên chỉ nội suy
    một lần thay vì ba lần chồng lên nhau.
    """
    n, dev = batch.size(0), batch.device
    flip = torch.rand(n, device=dev) < 0.5
    batch = torch.where(flip.view(-1, 1, 1, 1), batch.flip(-1), batch)

    rand = lambda: torch.rand(n, device=dev) * 2 - 1          # noqa: E731  -1..1
    angles = rand() * (cfg["rotation"] * np.pi / 180)
    zoom = 1 + rand() * cfg["scale"]
    cos, sin = torch.cos(angles) * zoom, torch.sin(angles) * zoom
    theta = torch.zeros(n, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 1] = cos, -sin
    theta[:, 1, 0], theta[:, 1, 1] = sin, cos
    theta[:, 0, 2] = rand() * cfg["translate"]
    theta[:, 1, 2] = rand() * cfg["translate"]
    grid = F.affine_grid(theta, batch.shape, align_corners=False)
    batch = F.grid_sample(batch, grid, align_corners=False, padding_mode="zeros")

    j = cfg["jitter"]
    scale = 1 + rand().view(-1, 1, 1, 1) * j
    contrast = 1 + rand().view(-1, 1, 1, 1) * j
    mean = batch.mean(dim=(1, 2, 3), keepdim=True)
    return ((batch * scale - mean) * contrast + mean).clamp(0, 1)


def to_model_input(batch_uint8, size=IMG_SIZE, aug=None):
    """(B,H,W) uint8 -> (B,3,H,W) chuẩn hoá trên DEVICE.

    Cache giữ ảnh ở IMG_SIZE; thí nghiệm nào cần kích thước khác thì thu nhỏ
    ngay trên DEVICE. Đây là resize hai bước (gốc -> IMG_SIZE -> size), áp dụng
    đồng nhất cho mọi split nên không tạo chênh lệch giữa train và test.
    """
    x = batch_uint8.to(DEVICE, non_blocking=True).float().div_(255).unsqueeze(1)
    if size != IMG_SIZE:
        x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    if aug is not None:
        x = device_augment(x, aug)
    return (x.expand(-1, 3, -1, -1) - MEAN_T) / STD_T


def resize_for_cache(image, size=IMG_SIZE, mode=RESIZE_MODE):
    gray = image.convert("L")
    if mode == "stretch":
        return np.asarray(gray.resize((size, size), Image.Resampling.BILINEAR))
    if mode != "letterbox":
        raise ValueError(f"RESIZE_MODE lạ: {mode}")
    gray.thumbnail((size, size), Image.Resampling.BILINEAR)
    array = np.asarray(gray)
    fill = int(np.median(array))
    canvas = Image.new("L", (size, size), color=fill)
    offset = ((size - gray.width) // 2, (size - gray.height) // 2)
    canvas.paste(gray, offset)
    return np.asarray(canvas)


def build_image_cache(manifest, size=IMG_SIZE, mode=RESIZE_MODE):
    cache = np.zeros((len(manifest), size, size), dtype=np.uint8)
    for position, path in enumerate(manifest["path"]):
        with Image.open(path) as image:
            cache[position] = resize_for_cache(image, size, mode)
        if (position + 1) % 1500 == 0:
            print(f"  {position + 1:,}/{len(manifest):,}")
    return cache


# Mỗi chế độ resize cần một cache riêng. Chỉ dựng những chế độ thực sự được
# dùng, để so sánh stretch với letterbox nằm trong cùng một lần chạy thay vì hai
# lần chạy khác nhau như trước.
REQUIRED_MODES = sorted({spec.get("resize", RESIZE_MODE) for spec in EXPERIMENTS})
IMAGE_CACHES = {}
for mode in REQUIRED_MODES:
    _started = time.time()
    IMAGE_CACHES[mode] = build_image_cache(manifest, mode=mode)
    log(f"cache {mode}: {IMAGE_CACHES[mode].nbytes / 1e6:.0f} MB cho "
        f"{len(manifest):,} ảnh trong {time.time() - _started:.0f}s")

# Cache mặc định cho các đoạn không gắn với một thí nghiệm cụ thể.
IMAGE_CACHE = IMAGE_CACHES[RESIZE_MODE if RESIZE_MODE in IMAGE_CACHES
                           else REQUIRED_MODES[0]]


class XRayDataset(Dataset):
    """Trả về ảnh uint8 thô; augmentation diễn ra trên DEVICE."""

    def __init__(self, rows, mode=RESIZE_MODE):
        self.rows, self.mode = rows, mode

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        cache_index, label = self.rows[index]
        # Stage B also returns the cache index so the training loop can look up
        # which images sit in a hard group.
        return (cache_index,
                torch.from_numpy(IMAGE_CACHES[self.mode][cache_index]), label)


def make_loader(split, name, batch_size=BATCH_SIZE, seed=SEED, mode=RESIZE_MODE):
    subset = split[split["split"] == name]
    return DataLoader(
        XRayDataset(list(zip(subset["cache_index"], subset["class_id"])), mode),
        batch_size=batch_size, shuffle=(name == "train"),
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
        **loader_seed_args(seed))


def make_loaders(split, batch_size=BATCH_SIZE, seed=SEED, mode=RESIZE_MODE):
    return {name: make_loader(split, name, batch_size, seed, mode)
            for name in ("train", "val")}


def class_weights_from(split):
    """Trọng số nghịch tần suất, chuẩn hoá để loss giữ nguyên thang đo."""
    counts = Counter(split[split["split"] == "train"]["class_id"])
    total = sum(counts.values())
    return torch.tensor([total / (len(CLASSES) * counts[i]) for i in range(len(CLASSES))],
                        dtype=torch.float, device=DEVICE)

  1,500/5,856
  3,000/5,856
  4,500/5,856
cache stretch: 294 MB cho 5,856 ảnh trong 52s


## 2.3. Chỉ số đánh giá

In [10]:
METRIC_COLS = ["accuracy", "precision", "recall", "specificity",
               "f1", "bal_acc", "auc", "pr_auc"]


def metrics_at(labels, probs, threshold=0.5):
    """Chấm điểm tại một ngưỡng. threshold=0.5 chính là argmax trên hai logit."""
    labels, probs = np.asarray(labels), np.asarray(probs)
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    sensitivity, specificity = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    return {
        "threshold": float(threshold),
        "accuracy": float((labels == preds).mean()),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "specificity": float(specificity),
        "f1": f1_score(labels, preds, zero_division=0),
        "bal_acc": float((sensitivity + specificity) / 2),
        "auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs),
        "confusion_matrix": confusion_matrix(labels, preds).tolist(),
    }


def to_group_level(group_ids, labels, probs):
    """Gộp theo filename-derived group; xác suất là trung bình các ảnh."""
    frame = pd.DataFrame({"group": group_ids, "label": labels, "prob": probs})
    label_counts = frame.groupby("group")["label"].nunique()
    if int(label_counts.max()) != 1:
        bad = label_counts[label_counts > 1].index.tolist()[:5]
        raise ValueError(f"Group chứa nhiều nhãn, ví dụ: {bad}")
    rolled = frame.groupby("group", sort=True).agg(
        label=("label", "first"), prob=("prob", "mean"))
    return rolled["label"].to_numpy(), rolled["prob"].to_numpy()


def tune_threshold(labels, probs, objective=THRESHOLD_OBJECTIVE,
                   target_sensitivity=TARGET_SENSITIVITY):
    """Chọn một candidate thật trên validation/OOF, không nội suy qua vùng tie."""
    labels, probs = np.asarray(labels), np.asarray(probs)
    candidates = np.unique(np.clip(probs, 0.001, 0.999))
    if len(candidates) > 400:
        indices = np.linspace(0, len(candidates) - 1, 400).round().astype(int)
        candidates = candidates[np.unique(indices)]

    rows = [metrics_at(labels, probs, float(t)) for t in candidates]
    if objective == "balanced_accuracy":
        best = max(rows, key=lambda m: (m["bal_acc"], m["specificity"],
                                        m["recall"], m["threshold"]))
    elif objective == "sensitivity":
        feasible = [m for m in rows if m["recall"] >= target_sensitivity - 1e-12]
        if not feasible:
            warnings.warn("Không có ngưỡng đạt target sensitivity; dùng recall cao nhất.")
            feasible = rows
            best = max(feasible, key=lambda m: (m["recall"], m["specificity"],
                                                m["threshold"]))
        else:
            best = max(feasible, key=lambda m: (m["specificity"], m["bal_acc"],
                                                m["threshold"]))
    else:
        raise ValueError(f"THRESHOLD_OBJECTIVE lạ: {objective}")
    return float(best["threshold"]), best

## 2.4. Kiến trúc

In [11]:
class SmallCNN(nn.Module):
    """4 khối Conv-BN-ReLU-Pool rồi gộp toàn cục. Train từ đầu."""

    def __init__(self, num_classes=len(CLASSES)):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64),
                                      block(64, 128), block(128, 256))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model(arch, pretrained=True, device=DEVICE):
    if arch == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    elif arch == "densenet121":
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        model = models.densenet121(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features, len(CLASSES))
    elif arch == "small_cnn":
        model = SmallCNN()
    else:
        raise ValueError(f"Kiến trúc lạ: {arch!r}")
    return model.to(device)


def target_layer_for(model, arch):
    """Lớp tích chập cuối để gắn Grad-CAM. Chỉ đích danh theo kiến trúc.

    Cách dò 'Conv2d cuối cùng' sẽ sai âm thầm khi đổi kiến trúc — heatmap vẫn
    hiện ra, chỉ là hiện sai chỗ.
    """
    return {"resnet18": lambda: model.layer4[-1],
            "densenet121": lambda: model.features.denseblock4,
            "small_cnn": lambda: model.features[-1]}[arch]()


def parameter_count_millions(arch):
    model = build_model(arch, pretrained=False, device=torch.device("cpu"))
    count = sum(p.numel() for p in model.parameters()) / 1e6
    del model
    return round(count, 1)


display(pd.DataFrame([
    {"thí nghiệm": s["name"], "kiến trúc": s["arch"], "px": s["size"],
     "resize": s.get("resize", RESIZE_MODE),
     "augment": s["aug"], "balancing": s["balancing"],
     "tham số (M)": parameter_count_millions(s["arch"]),
     "câu hỏi": s["hoi"]}
    for s in EXPERIMENTS]).set_index("thí nghiệm"))

,kiến trúc,px,resize,augment,balancing,tham số (M),câu hỏi
thí nghiệm,,,,,,,
densenet_hn2,densenet121,224,stretch,manh,weighted,7.0,DenseNet121 + trọng số nhóm NORMAL khó


## 2.5. Nguồn gốc checkpoint

Trước khi chạm vào dữ liệu, xác định chính xác checkpoint nào đang được dùng và
băm chúng lại.

In [12]:
def find_source_root():
    """Locate the attached v5 notebook output without hardcoding its slug.

    Kaggle names the mount after the source notebook, and that name changes if
    the notebook is renamed or copied. Searching for the checkpoints themselves
    survives that; requiring exactly one file per fold catches the case where
    two versions of the output are attached at once.

    Returns:
        Mapping of fold index to checkpoint path.

    Raises:
        RuntimeError: If the folds are not exactly 0 to 4, or one is ambiguous.
    """
    roots = [Path("/kaggle/input")] if IS_KAGGLE else [
        LOCAL_PROJECT_ROOT / "notebooks/results_v5"]
    found = {}
    for root in roots:
        if not root.is_dir():
            continue
        for path in root.rglob("*"):
            if path.suffix.lower() not in {".pt", ".pth"}:
                continue
            if SOURCE_EXPERIMENT not in path.stem:
                continue
            match = re.search(r"fold[_-]?([0-4])", path.stem, re.IGNORECASE)
            if match is None:
                continue
            fold = int(match.group(1))
            if fold in found:
                raise RuntimeError(
                    f"Nhiều checkpoint cho fold {fold}: {found[fold]} và {path}. "
                    "Gỡ bớt input để chỉ còn đúng một version của v5.")
            found[fold] = path
    if set(found) != set(range(5)):
        raise RuntimeError(
            f"Cần đủ fold 0-4, chỉ thấy {sorted(found)}. Trên Kaggle hãy "
            "Add Input -> Your Work -> Notebook -> bản v5 chạy THÀNH CÔNG.")
    return dict(sorted(found.items()))


def find_source_file(pattern, root=None):
    """Find one supporting file, scoped to avoid picking up the wrong run.

    Several runs are attached at once and they share filenames, so an
    unscoped search matches v4's resolved_config.json as readily as v5's.
    Files belonging to the checkpoint source are looked up under the directory
    the checkpoints came from; anything else searches every input.

    Args:
        pattern: Glob pattern matched against filenames.
        root: Directory to search, or None to search all inputs.

    Returns:
        The single matching path.

    Raises:
        RuntimeError: If the pattern matches other than exactly one file.
    """
    if root is not None:
        roots = [root]
    else:
        roots = [Path("/kaggle/input")] if IS_KAGGLE else [
            LOCAL_PROJECT_ROOT / "notebooks"]
    hits = sorted({p for r in roots if r.is_dir() for p in r.rglob(pattern)})
    if len(hits) != 1:
        listing = ", ".join(str(h) for h in hits[:4])
        raise RuntimeError(
            f"{pattern}: cần đúng 1 file, thấy {len(hits)}. {listing}")
    return hits[0]


CHECKPOINTS = find_source_root()
SOURCE_ROOT = CHECKPOINTS[0].parent
log(f"nguồn v5: {SOURCE_ROOT}")
log("checkpoint nguồn:")
for fold, path in CHECKPOINTS.items():
    digest = hashlib.sha256(path.read_bytes()).hexdigest()[:16]
    log(f"  fold {fold}: {path.name}  sha256 {digest}  "
        f"{path.stat().st_size / 1e6:.1f} MB")

source_config = json.loads(
    find_source_file("resolved_config.json", SOURCE_ROOT).read_text())
source_spec = next(s for s in source_config["experiments"]
                   if s["name"] == SOURCE_EXPERIMENT)
actual = {k: source_spec[k] for k in EXPECTED_SOURCE}
if actual != EXPECTED_SOURCE:
    raise RuntimeError(f"Cấu hình nguồn lệch: {actual}; kỳ vọng {EXPECTED_SOURCE}")
log(f"\ncấu hình nguồn khớp: {actual}")

nguồn v5: /kaggle/input/notebooks/vntinphan/model-improvement-pneumonia
checkpoint nguồn:
  fold 0: densenet121_robust_fold0.pth  sha256 38bb6c8c25b9383c  28.4 MB
  fold 1: densenet121_robust_fold1.pth  sha256 c5a73086183af9ab  28.4 MB
  fold 2: densenet121_robust_fold2.pth  sha256 1742ea09d41fb0fd  28.4 MB
  fold 3: densenet121_robust_fold3.pth  sha256 19e64ef12d104e7a  28.4 MB
  fold 4: densenet121_robust_fold4.pth  sha256 a15598e21fb9d9be  28.4 MB

cấu hình nguồn khớp: {'arch': 'densenet121', 'size': 224, 'resize': 'stretch', 'aug': 'manh', 'balancing': 'weighted'}


## 2.5b. Dùng lại đúng cách chia của v5

Không dựng lại fold. `StratifiedGroupKFold` phụ thuộc thứ tự hàng trong
manifest, và thứ tự đó phụ thuộc cách hệ tệp liệt kê ảnh — nên fold dựng lại
trên một máy khác có thể lệch vài chục ảnh so với bản gốc. Lệch một ảnh là đủ
để epoch 0 không còn so được với dự đoán đã lưu.

v5 đã lưu năm manifest. Đọc thẳng chúng loại bỏ hẳn nguồn sai lệch này.

In [13]:
_local = dict(zip(manifest["filename"], manifest["path"]))
_cache_index = dict(zip(manifest["filename"], manifest["cache_index"]))
FOLDS = []
for _index in range(5):
    _saved = pd.read_csv(find_source_file(f"manifest_fold{_index}.csv",
                                          SOURCE_ROOT))
    # Đường dẫn trong manifest v5 trỏ tới mount của Kaggle lúc đó; ánh xạ lại
    # theo tên file để notebook chạy được ở bất kỳ đâu.
    _saved["path"] = _saved["filename"].map(_local)
    if _saved["path"].isna().any():
        _missing = _saved.loc[_saved["path"].isna(), "filename"].head(3).tolist()
        raise RuntimeError(f"manifest v5 có ảnh không tìm thấy: {_missing}")
    _saved["cache_index"] = _saved["filename"].map(_cache_index)
    FOLDS.append(_saved)

log(f"dùng lại {len(FOLDS)} manifest của v5")
for _index, _fold in enumerate(FOLDS):
    _counts = _fold["split"].value_counts()
    _groups = _fold.groupby("split")["group_id"].nunique()
    log(f"  fold {_index}: train {_counts.get('train', 0):,} "
        f"val {_counts.get('val', 0):,} test {_counts.get('test', 0):,}  |  "
        f"group train/val chồng nhau: "
        f"{len(set(_fold[_fold.split == 'train'].group_id) & set(_fold[_fold.split == 'val'].group_id))}")
    assert not (set(_fold[_fold.split == "train"].group_id)
                & set(_fold[_fold.split == "val"].group_id))

dùng lại 5 manifest của v5
  fold 0: train 4,168 val 1,064 test 624  |  group train/val chồng nhau: 0
  fold 1: train 4,224 val 1,008 test 624  |  group train/val chồng nhau: 0
  fold 2: train 4,165 val 1,067 test 624  |  group train/val chồng nhau: 0
  fold 3: train 4,190 val 1,042 test 624  |  group train/val chồng nhau: 0
  fold 4: train 4,181 val 1,051 test 624  |  group train/val chồng nhau: 0


## 2.6. Bảng nhóm NORMAL khó

Dựng lại từ dự đoán out-of-fold của hai mô hình đã đóng băng, rồi đối chiếu băm
với bản đã khóa. Rank tính **trong training split của từng fold** — nếu tính
trên pool chung thì cả năm cutoff sẽ bằng nhau, và notebook dừng.

In [14]:
def build_hardness(teachers, manifest, fraction=HARD_FRACTION):
    """Rank one fold's training normals by how hard the teachers found them.

    Ranks are computed inside this fold's own training normals. Ranking over a
    shared pool would let groups the fold never trains on move its cutoff, and
    all five folds would then land on the same number.

    Args:
        teachers: Merged out-of-fold predictions, one row per group.
        manifest: This fold's manifest with group_id and split.
        fraction: Portion of normals to mark hard.

    Returns:
        Table of this fold's training normals with hardness and flag.
    """
    training = set(manifest.loc[manifest["split"] == "train", "group_id"])
    block = teachers[teachers["group_id"].isin(training)
                     & (teachers["label"] == 0)].copy()
    if block.empty:
        raise ValueError("Fold không có group NORMAL nào trong training split")
    columns = [c for c in block.columns if c.startswith("p_")]
    for column in columns:
        block[f"rank_{column[2:]}"] = (
            block[column].rank(method="average") / (len(block) + 1.0))
    block["hardness"] = block[[f"rank_{c[2:]}" for c in columns]].mean(axis=1)
    n_hard = int(round(fraction * len(block)))
    cutoff = block["hardness"].nlargest(n_hard).min() if n_hard else np.inf
    block["hard_normal"] = block["hardness"] >= cutoff
    return block.sort_values("hardness", ascending=False).reset_index(drop=True)


def load_teacher(experiment, name):
    """Read one model's pooled out-of-fold group predictions.

    v5 writes a pooled file directly; v4 predates it and only has per-fold
    validation predictions, so those are pooled here. Either way every group
    is scored exactly once, by the fold that held it out.

    Args:
        experiment: Experiment name as it appears in filenames.
        name: Short model name used to prefix its probability column.

    Returns:
        Frame with group_id, label and a prefixed probability column.

    Raises:
        ValueError: If a group appears more than once.
    """
    try:
        frame = pd.read_csv(find_source_file(
            f"predictions_oof_{experiment}_groups.csv"))
        frame = frame[["group_id", "label", "p_pneumonia"]]
    except RuntimeError:
        parts = []
        for fold in range(5):
            path = find_source_file(
                f"validation_predictions_{experiment}_fold{fold}.csv")
            parts.append(pd.read_csv(
                path, usecols=["group_id", "class_id", "p_pneumonia"]))
        pooled = pd.concat(parts, ignore_index=True)
        frame = (pooled.groupby("group_id", as_index=False)
                 .agg(label=("class_id", "first"),
                      p_pneumonia=("p_pneumonia", "mean")))
    if frame["group_id"].duplicated().any():
        raise ValueError(f"{name}: group_id lặp lại")
    return frame.rename(columns={"p_pneumonia": f"p_{name}"})


resnet = load_teacher("stretch_manh", "resnet")
densenet = load_teacher("densenet121_robust", "densenet")
TEACHERS = resnet.merge(densenet, on=["group_id", "label"], how="inner")
if not (len(TEACHERS) == len(resnet) == len(densenet)):
    raise RuntimeError("Hai bảng OOF không cùng tập group")
log(f"teacher: {len(TEACHERS):,} group, "
    f"{int((TEACHERS['label'] == 0).sum()):,} NORMAL")

HARDNESS = {}
rows = []
for index, manifest in enumerate(FOLDS):
    table = build_hardness(TEACHERS, manifest)
    HARDNESS[index] = table
    hard, rest = table[table["hard_normal"]], table[~table["hard_normal"]]

    # Leakage assertions, run every time rather than trusted.
    training = set(manifest.loc[manifest["split"] == "train", "group_id"])
    other = set(manifest.loc[manifest["split"] != "train", "group_id"])
    assert set(table["group_id"]) <= training, f"fold {index}: có group ngoài train"
    assert not (set(table["group_id"]) & other), f"fold {index}: rò rỉ val/test"
    assert (table["label"] == 0).all(), f"fold {index}: có PNEUMONIA bị đánh dấu"
    assert not table["group_id"].duplicated().any(), f"fold {index}: group trùng"

    path = WORK_DIR / f"hard_negative_groups_fold{index}.csv"
    table.to_csv(path, index=False)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()[:16]
    rows.append({"fold": index, "n_train_normal_groups": len(table),
                 "n_hard_normal_groups": int(len(hard)),
                 "hard_fraction": len(hard) / len(table),
                 "hardness_cutoff": float(table.loc[table["hard_normal"],
                                                    "hardness"].min()),
                 "median_resnet_hard": float(hard["p_resnet"].median()),
                 "median_resnet_other": float(rest["p_resnet"].median()),
                 "median_densenet_hard": float(hard["p_densenet"].median()),
                 "median_densenet_other": float(rest["p_densenet"].median()),
                 "sha256": digest})

hardness_summary = pd.DataFrame(rows)
hardness_summary.to_csv(WORK_DIR / "hard_negative_summary.csv", index=False)
display(hardness_summary.round(4))

if hardness_summary["hardness_cutoff"].nunique() != len(FOLDS):
    raise RuntimeError("Các fold có cutoff giống nhau: rank có thể đã tính "
                       "trên pool chung thay vì trong từng training split.")

mismatched = {f"fold{r.fold}": (r.sha256, EXPECTED_HARDNESS_HASHES[f"fold{r.fold}"])
              for r in hardness_summary.itertuples()
              if len(FOLDS) == 5
              and r.sha256 != EXPECTED_HARDNESS_HASHES[f"fold{r.fold}"]}
if mismatched:
    log("\nCẢNH BÁO: bảng hardness khác bản đã khóa ở commit b3036dd:")
    for fold, (got, want) in mismatched.items():
        log(f"  {fold}: dựng lại {got}  đã khóa {want}")
    log("  Nếu đây là smoke run với ít fold hơn thì bỏ qua; nếu là full run "
        "thì dữ liệu nguồn đã thay đổi.")
else:
    log("\nbảng hardness khớp bản đã khóa ở commit b3036dd")

teacher: 3,669 group, 1,219 NORMAL


,fold,n_train_normal_groups,n_hard_normal_groups,hard_fraction,hardness_cutoff,median_resnet_hard,median_resnet_other,median_densenet_hard,median_densenet_other,sha256
0,0,976,244,0.2500,0.7160,0.0102,0.0001,0.0232,0.0003,23117298e5a45623
1,1,975,244,0.2503,0.7075,0.0148,0.0001,0.0187,0.0002,c457d92a83b7df20
2,2,975,244,0.2503,0.7223,0.0153,0.0001,0.0152,0.0001,ef1fe49e2f2537b7
3,3,975,244,0.2503,0.7141,0.0107,0.0001,0.0190,0.0001,732b2b3c5fe04e3f
4,4,975,244,0.2503,0.7085,0.0075,0.0001,0.0214,0.0001,956fc326994fd853



CẢNH BÁO: bảng hardness khác bản đã khóa ở commit b3036dd:
  fold0: dựng lại 23117298e5a45623  đã khóa b6c6a6a3d8891724
  fold1: dựng lại c457d92a83b7df20  đã khóa 25081fc3411338f3
  fold2: dựng lại ef1fe49e2f2537b7  đã khóa 068bea3a87521f51
  fold3: dựng lại 732b2b3c5fe04e3f  đã khóa 5bcf6bf8fb177992
  fold4: dựng lại 956fc326994fd853  đã khóa 2c7bbc70c79d15c0
  Nếu đây là smoke run với ít fold hơn thì bỏ qua; nếu là full run thì dữ liệu nguồn đã thay đổi.


## 2.7. Vòng fine-tune

Mẫu số của loss là tổng trọng số **kết hợp** (trọng số lớp × multiplier), không
phải tổng multiplier. PyTorch đã chuẩn hóa weighted cross-entropy theo tổng
trọng số lớp chứ không theo cỡ batch, nên chia cho riêng multiplier sẽ đổi scale
loss ngay cả khi multiplier toàn 1 — và đổi scale loss là đổi learning rate hiệu
dụng, đúng thứ cần tránh.

In [15]:
@torch.no_grad()
def predict(model, loader, size=IMG_SIZE):
    """(nhãn thật, xác suất PNEUMONIA) trên toàn bộ loader."""
    model.eval()
    labels_all, probs_all = [], []
    for _, images, labels in loader:
        logits = model(to_model_input(images, size))
        probs_all += torch.softmax(logits.float(), dim=1)[:, 1].cpu().tolist()
        labels_all += labels.tolist()
    return np.array(labels_all), np.array(probs_all)


def group_scores(labels, probs, groups):
    """Gộp dự đoán về mức filename-derived group."""
    frame = pd.DataFrame({"g": groups, "y": labels, "p": probs})
    label_counts = frame.groupby("g")["y"].nunique()
    if int(label_counts.max()) != 1:
        bad = label_counts[label_counts > 1].index.tolist()[:5]
        raise ValueError(f"Group validation chứa nhiều nhãn, ví dụ: {bad}")
    rolled = frame.groupby("g", sort=True).agg(y=("y", "first"), p=("p", "mean"))
    return rolled["y"].to_numpy(), rolled["p"].to_numpy()


def specificity_at_sensitivity(labels, probs, target=TARGET_SENSITIVITY):
    """Độ đặc hiệu cao nhất còn giữ được độ nhạy tối thiểu, kèm ngưỡng.

    Đây là chỉ số dự án đang thực sự cần cải thiện. Validation AUC đã bão hòa ở
    0,999x nên chọn epoch theo nó chỉ là chọn theo nhiễu.
    """
    positive = probs[labels == 1]
    if not len(positive):
        return 0.0, 0.5
    feasible = [c for c in np.unique(probs) if (positive >= c).mean() >= target]
    if not feasible:
        return 0.0, 0.0
    threshold = float(max(feasible))
    return float((probs[labels == 0] < threshold).mean()), threshold


def group_nll(labels, probs):
    """Log-loss không trọng số ở mức group.

    Không dùng loss có trọng số lớp để đánh giá: trọng số làm lệch thang xác
    suất, nên nó không nói được mô hình hiệu chuẩn tốt hay xấu.
    """
    p = np.clip(probs, 1e-7, 1 - 1e-7)
    return float(-np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p)))


def group_brier(labels, probs):
    """Brier score ở mức group."""
    return float(np.mean((probs - labels) ** 2))


def better_checkpoint(candidate, incumbent, margin=CHECKPOINT_TIE_MARGIN):
    """Quy tắc chọn checkpoint, khóa trước khi chạy.

    Độ đặc hiệu là chỉ số chính. Chênh lệch dưới ``margin`` coi như bằng nhau và
    NLL quyết định, vì độ đặc hiệu ở mức group nhảy bậc rời rạc — một ca đổi
    phía đã là 0,004 — nên chênh lệch nhỏ hơn thế là nhiễu lấy mẫu.

    Args:
        candidate: Chỉ số của epoch hiện tại.
        incumbent: Chỉ số của checkpoint đang giữ, hoặc None.
        margin: Ngưỡng coi hai độ đặc hiệu là bằng nhau.

    Returns:
        True nếu nên thay checkpoint.
    """
    if incumbent is None:
        return True
    gap = candidate["specificity"] - incumbent["specificity"]
    if gap > margin + 1e-12:
        return True
    if gap < -margin - 1e-12:
        return False
    # Hòa về độ đặc hiệu: lấy NLL thấp hơn. Vẫn hòa thì giữ epoch sớm hơn.
    return candidate["nll"] < incumbent["nll"] - 1e-9


def partial_auc(labels, probs, min_sensitivity=TARGET_SENSITIVITY):
    """ROC area restricted to the high-sensitivity region, rescaled to [0, 1].

    Global AUC integrates the whole curve and barely registers a few dozen
    normals reordered near the operating point, which is exactly where this
    model is used and exactly what Stage B is trying to move. Logged per epoch
    but deliberately not used for checkpoint selection, so hard-negative
    weighting stays the only methodological change.

    Args:
        labels: Binary labels.
        probs: Predicted probabilities.
        min_sensitivity: Lower bound of the region of interest.

    Returns:
        Normalised partial AUC, or NaN when the region is degenerate.
    """
    fpr, tpr, _ = roc_curve(labels, probs)
    mask = tpr >= min_sensitivity
    if mask.sum() < 2:
        return float("nan")
    x, y = fpr[mask], tpr[mask]
    width = x.max() - x.min()
    if width < 1e-12:
        return float("nan")
    return float(np.trapezoid(y - min_sensitivity, x)
                 / (width * (1.0 - min_sensitivity)))

In [16]:
def hard_negative_loss(logits, labels, class_weights, multiplier):
    """Weighted cross-entropy that leans on the flagged normals.

    The denominator is the summed combined weight, not the summed multipliers.
    PyTorch's weighted cross-entropy already normalises by the summed class
    weights rather than by the batch size, so dividing by multipliers alone
    rescales the loss even when every multiplier is one, and rescaling the loss
    rescales the effective learning rate. Emphasis is the intended change; step
    size is not.

    Args:
        logits: Model outputs.
        labels: Target classes.
        class_weights: Per-class weights, or None.
        multiplier: Per-sample multiplier.

    Returns:
        Scalar loss.
    """
    per_sample = F.cross_entropy(logits, labels, weight=class_weights,
                                 reduction="none")
    weight_per_sample = (torch.ones_like(multiplier) if class_weights is None
                         else class_weights[labels])
    combined = weight_per_sample * multiplier
    return (per_sample * multiplier).sum() / combined.sum().clamp_min(1e-12)


def load_source_model(fold_index):
    """Rebuild a frozen v5 checkpoint without touching ImageNet.

    The state dict is the only source of weights, so downloading pretrained
    parameters would be both unnecessary and a way for a different version to
    creep in. strict=True makes any architecture mismatch fail here.

    Args:
        fold_index: Which fold's checkpoint to load.

    Returns:
        The model on DEVICE, in eval mode.
    """
    model = models.densenet121(weights=None)
    model.classifier = nn.Linear(model.classifier.in_features, len(CLASSES))
    state = torch.load(CHECKPOINTS[fold_index], map_location="cpu",
                       weights_only=True)
    model.load_state_dict(state, strict=True)
    return model.to(DEVICE).eval()


def check_epoch_zero(fold_index, labels, probs, rows):
    """Confirm the loaded checkpoint reproduces the run it came from.

    If this fails the weights are not the ones that produced v5's numbers, and
    every comparison against v5 downstream would be meaningless.

    Args:
        fold_index: Fold being checked.
        labels: Labels predicted over, in manifest order.
        probs: Freshly computed probabilities.
        rows: Manifest rows for this fold's validation split.

    Returns:
        Mapping describing the agreement.

    Raises:
        RuntimeError: If the saved predictions cannot be located or disagree.
    """
    saved = pd.read_csv(find_source_file(
        f"validation_predictions_{SOURCE_EXPERIMENT}_fold{fold_index}.csv",
        SOURCE_ROOT))
    if not np.array_equal(saved["filename"].to_numpy(),
                          rows["filename"].to_numpy()):
        raise RuntimeError(f"fold {fold_index}: thứ tự ảnh khác bản v5")
    if not np.array_equal(saved["class_id"].to_numpy(), labels):
        raise RuntimeError(f"fold {fold_index}: nhãn khác bản v5")
    reference = saved["p_pneumonia"].to_numpy()
    difference = float(np.abs(probs - reference).max())
    correlation = float(np.corrcoef(probs, reference)[0, 1])
    entry = {"fold": fold_index, "n": len(probs), "max_abs_diff": difference,
             "correlation": correlation,
             "passes": difference <= EPOCH0_MAX_ABS_DIFF
                       and correlation >= EPOCH0_MIN_CORRELATION}
    rows.append if False else None
    log(f"  epoch 0 fold {fold_index}: lệch tối đa {difference:.3e}  "
        f"tương quan {correlation:.8f}  "
        f"{'ĐẠT' if entry['passes'] else 'HỎNG'}")
    if not entry["passes"]:
        raise RuntimeError(
            f"fold {fold_index}: epoch 0 không tái hiện được checkpoint v5 "
            f"(lệch {difference:.3e}, tương quan {correlation:.8f})")
    return entry


def run_fold(spec, fold_index, epochs=None):
    """Fine-tune one fold from its frozen checkpoint.

    Args:
        spec: Experiment specification.
        fold_index: Which fold to run.
        epochs: Fine-tuning epochs beyond epoch 0.

    Returns:
        Result mapping in the same shape v5 produced.
    """
    epochs = FINETUNE_EPOCHS if epochs is None else epochs
    log(f"\n{'=' * 62}\n{spec['name']}  |  fold {fold_index}\n{'=' * 62}")
    resize, size = spec["resize"], spec["size"]
    aug = AUG_PRESETS[spec["aug"]]
    set_seed(SEED + fold_index)

    split = FOLDS[fold_index]
    loaders = make_loaders(split, seed=SEED + fold_index, mode=resize)
    model = load_source_model(fold_index)

    val_rows = split[split["split"] == "val"].reset_index(drop=True)
    val_groups = val_rows["group_id"].to_numpy()
    train_rows = split[split["split"] == "train"].reset_index(drop=True)
    hard_groups = set(HARDNESS[fold_index].loc[
        HARDNESS[fold_index]["hard_normal"], "group_id"])
    hard_by_index = dict(zip(train_rows["cache_index"],
                             train_rows["group_id"].isin(hard_groups)))
    log(f"NORMAL khó: {len(hard_groups)} group, "
        f"{int(sum(hard_by_index.values())):,}/{len(train_rows):,} ảnh train")

    weights = (class_weights_from(split) if spec["balancing"] == "weighted"
               else None)
    optimizer = torch.optim.Adam(model.parameters(), lr=FINETUNE_LR,
                                 weight_decay=FINETUNE_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

    def evaluate(epoch, train_loss):
        labels, probs = predict(model, loaders["val"], size)
        g_labels, g_probs = group_scores(labels, probs, val_groups)
        specificity, threshold = specificity_at_sensitivity(g_labels, g_probs)
        operating = metrics_at(g_labels, g_probs, threshold)
        (tn, fp), (fn, tp) = operating["confusion_matrix"]
        return {"epoch": epoch, "train_loss": train_loss,
                "specificity": specificity,
                "sensitivity": operating["recall"], "threshold": threshold,
                "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
                "nll": group_nll(g_labels, g_probs),
                "brier": group_brier(g_labels, g_probs),
                "auc": roc_auc_score(g_labels, g_probs),
                "pr_auc": average_precision_score(g_labels, g_probs),
                "partial_auc_sens97": partial_auc(g_labels, g_probs)}, labels, probs

    # Epoch 0 is the untouched checkpoint and stays eligible: if fine-tuning
    # makes a fold worse, that fold should keep what it already had.
    zero, labels0, probs0 = evaluate(0, float("nan"))
    EPOCH_ZERO_CHECKS.append(check_epoch_zero(fold_index, labels0, probs0,
                                              val_rows))
    best, best_epoch, history = zero, 0, [zero]
    best_state = {k: v.detach().cpu().clone()
                  for k, v in model.state_dict().items()}
    log(f"epoch  0 (gốc)  spec@sens{TARGET_SENSITIVITY:.0%} "
        f"{zero['specificity']:.4f}  AUC {zero['auc']:.4f}  "
        f"NLL {zero['nll']:.4f}  pAUC {zero['partial_auc_sens97']:.4f}")

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for indices, images, labels in loaders["train"]:
            inputs = to_model_input(images, size, aug)
            labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
            multiplier = torch.tensor(
                [HARD_MULTIPLIER if hard_by_index.get(int(i), False) else 1.0
                 for i in indices], device=DEVICE, dtype=torch.float32)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                loss = hard_negative_loss(model(inputs), labels, weights,
                                          multiplier)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * inputs.size(0)

        current, _, _ = evaluate(epoch, running / len(loaders["train"].dataset))
        history.append(current)
        marker = ""
        if better_checkpoint(current, best):
            best, best_epoch = current, epoch
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
            marker = "  <- best"
        log(f"epoch {epoch:>2}/{epochs}  loss {current['train_loss']:.4f}  "
            f"spec@sens{TARGET_SENSITIVITY:.0%} {current['specificity']:.4f}  "
            f"sens {current['sensitivity']:.4f}  AUC {current['auc']:.4f}  "
            f"NLL {current['nll']:.4f}  pAUC "
            f"{current['partial_auc_sens97']:.4f}{marker}")

    model.load_state_dict(best_state)
    log(f"giữ checkpoint epoch {best_epoch}"
        f"{' (không đổi so với v5)' if best_epoch == 0 else ''}, "
        f"spec {best['specificity']:.4f}, NLL {best['nll']:.4f}")

    val_labels, val_probs = predict(model, loaders["val"], size)
    tag = f"{spec['name']}_fold{fold_index}"
    torch.save(best_state, WORK_DIR / f"{tag}.pth")
    val_rows.assign(p_pneumonia=val_probs).to_csv(
        WORK_DIR / f"validation_predictions_{tag}.csv", index=False)
    pd.DataFrame(history).to_csv(WORK_DIR / f"epoch_history_{tag}.csv",
                                 index=False)

    result = {"experiment": spec["name"], "resize": resize, "arch": spec["arch"],
              "size": size, "balancing": spec["balancing"], "fold": fold_index,
              "best_epoch": best_epoch, "kept_original": best_epoch == 0,
              "checkpoint": str(WORK_DIR / f"{tag}.pth"),
              "val_labels": val_labels, "val_probs": val_probs,
              "val_groups": val_groups,
              "val": metrics_at(val_labels, val_probs, 0.5), "selection": best}
    del model, loaders, optimizer, scaler, best_state
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()
    return result


EPOCH_ZERO_CHECKS = []

# 3. Kết quả

In [17]:
log(f"Bắt đầu {RUN_MODE}: {len(EXPERIMENTS)} thí nghiệm × {len(FOLDS)} fold × "
    f"tối đa {FINETUNE_EPOCHS} epoch fine-tune")
_t0 = time.time()
RUNS = [run_fold(spec, fold)
        for spec in EXPERIMENTS
        for fold in FOLDS_TO_RUN]
log(f"\ntổng thời gian: {(time.time() - _t0) / 60:.1f} phút "
    f"({len(EXPERIMENTS)} thí nghiệm × {len(FOLDS_TO_RUN)} fold)")

Bắt đầu full: 1 thí nghiệm × 5 fold × tối đa 3 epoch fine-tune

densenet_hn2  |  fold 0
NORMAL khó: 244 group, 286/4,168 ảnh train
  epoch 0 fold 0: lệch tối đa 1.110e-16  tương quan 1.00000000  ĐẠT
epoch  0 (gốc)  spec@sens97% 0.9918  AUC 0.9980  NLL 0.0873  pAUC 0.9480
epoch  1/3  loss 0.0194  spec@sens97% 0.9959  sens 0.9715  AUC 0.9977  NLL 0.1341  pAUC 0.9300
epoch  2/3  loss 0.0129  spec@sens97% 0.9918  sens 0.9715  AUC 0.9978  NLL 0.1234  pAUC 0.9370
epoch  3/3  loss 0.0101  spec@sens97% 0.9959  sens 0.9715  AUC 0.9975  NLL 0.1592  pAUC 0.9208
giữ checkpoint epoch 0 (không đổi so với v5), spec 0.9918, NLL 0.0873

densenet_hn2  |  fold 1
NORMAL khó: 244 group, 280/4,224 ảnh train
  epoch 0 fold 1: lệch tối đa 1.110e-16  tương quan 1.00000000  ĐẠT
epoch  0 (gốc)  spec@sens97% 0.9918  AUC 0.9991  NLL 0.0361  pAUC 0.9803
epoch  1/3  loss 0.0269  spec@sens97% 0.9959  sens 0.9714  AUC 0.9993  NLL 0.0741  pAUC 0.9812
epoch  2/3  loss 0.0147  spec@sens97% 0.9959  sens 0.9714  AUC 0.9993

In [18]:
display(pd.DataFrame([
    {"fold": r["fold"], "epoch giữ": r["best_epoch"],
     "giữ bản gốc": r["kept_original"],
     "spec@sens97": round(r["selection"]["specificity"], 4),
     "pAUC≥97": round(r["selection"]["partial_auc_sens97"], 4),
     "NLL": round(r["selection"]["nll"], 4)}
    for r in RUNS]).set_index("fold"))

kept = sum(r["kept_original"] for r in RUNS)
print(f"\n{kept}/{len(RUNS)} fold giữ nguyên checkpoint v5")
if kept >= 3:
    print("  Từ 3 fold trở lên giữ bản gốc là điều kiện an toàn đã đăng ký để")
    print("  cân nhắc multiplier 1.5 — quyết định trên tín hiệu OOF, không phải")
    print("  trên benchmark.")

display(pd.DataFrame(EPOCH_ZERO_CHECKS).set_index("fold"))

,epoch giữ,giữ bản gốc,spec@sens97,pAUC≥97,NLL
fold,,,,,
0,0,True,0.9918,0.9480,0.0873
1,0,True,0.9918,0.9803,0.0361
2,0,True,0.9959,0.9852,0.0432
3,0,True,0.9754,0.9423,0.0810
4,0,True,0.9959,0.9694,0.0563



5/5 fold giữ nguyên checkpoint v5
  Từ 3 fold trở lên giữ bản gốc là điều kiện an toàn đã đăng ký để
  cân nhắc multiplier 1.5 — quyết định trên tín hiệu OOF, không phải
  trên benchmark.


,n,max_abs_diff,correlation,passes
fold,,,,
0,1064,1.110223e-16,1.0,True
1,1008,1.110223e-16,1.0,True
2,1067,1.110223e-16,1.0,True
3,1042,1.110223e-16,1.0,True
4,1051,1.110223e-16,1.0,True


## 3.2. Tổng hợp OOF

In [19]:
DEV_LABEL = "OOF validation" if len(FOLDS) > 1 else "validation holdout"
OOF, rows = {}, []
for spec in EXPERIMENTS:
    runs = sorted((r for r in RUNS if r["experiment"] == spec["name"]),
                  key=lambda r: r["fold"])
    labels = np.concatenate([r["val_labels"] for r in runs])
    probs = np.concatenate([r["val_probs"] for r in runs])
    groups = np.concatenate([r["val_groups"] for r in runs])
    folds_oof = np.concatenate([
        np.full(len(r["val_labels"]), r["fold"], dtype=int) for r in runs])
    group_labels, group_probs = to_group_level(groups, labels, probs)
    oof_images = pd.DataFrame({
        "group_id": groups, "label": labels, "p_pneumonia": probs,
        "fold": folds_oof, "experiment": spec["name"],
    })
    oof_images.to_csv(
        WORK_DIR / f"predictions_oof_{spec['name']}_images.csv", index=False)
    oof_groups = (oof_images.groupby("group_id", as_index=False)
                  .agg(label=("label", "first"),
                       p_pneumonia=("p_pneumonia", "mean"),
                       fold=("fold", "first")))
    if oof_groups["group_id"].duplicated().any():
        raise AssertionError("OOF group bị lặp sau khi gộp.")
    oof_groups["experiment"] = spec["name"]
    oof_groups.to_csv(
        WORK_DIR / f"predictions_oof_{spec['name']}_groups.csv", index=False)

    image_threshold, image_tuned = tune_threshold(labels, probs)
    group_threshold, group_tuned = tune_threshold(group_labels, group_probs)
    entry = {
        "labels": labels, "probs": probs, "groups": groups,
        "group_labels": group_labels, "group_probs": group_probs,
        "image_threshold": image_threshold, "group_threshold": group_threshold,
        "image_default": metrics_at(labels, probs, 0.5),
        "image_tuned": image_tuned,
        "group_default": metrics_at(group_labels, group_probs, 0.5),
        "group_tuned": group_tuned,
    }
    OOF[spec["name"]] = entry
    for unit, default_key, tuned_key in (
            ("image", "image_default", "image_tuned"),
            ("filename_group", "group_default", "group_tuned")):
        for mode, key in (("default_0.5", default_key), ("validation_tuned", tuned_key)):
            block = entry[key]
            rows.append({"experiment": spec["name"],
                         "resize": spec.get("resize", RESIZE_MODE),
                         "aug": spec["aug"], "balancing": spec["balancing"],
                         "arch": spec["arch"], "size": spec["size"],
                         "unit": unit, "mode": mode,
                         "threshold": block["threshold"],
                         **{k: block[k] for k in METRIC_COLS}})

oof_results = pd.DataFrame(rows)
oof_results.to_csv(WORK_DIR / "results_oof_validation.csv", index=False)
print(f"{DEV_LABEL.upper()} — không dùng test để chọn cấu hình\n")
display(oof_results.set_index(["experiment", "unit", "mode"]).round(4))

OOF VALIDATION — không dùng test để chọn cấu hình



resize   aug balancing  \
experiment   unit           mode                                        
densenet_hn2 image          default_0.5       stretch  manh  weighted   
                            validation_tuned  stretch  manh  weighted   
             filename_group default_0.5       stretch  manh  weighted   
                            validation_tuned  stretch  manh  weighted   

                                                     arch  size  threshold  \
experiment   unit           mode                                             
densenet_hn2 image          default_0.5       densenet121   224     0.5000   
                            validation_tuned  densenet121   224     0.7020   
             filename_group default_0.5       densenet121   224     0.5000   
                            validation_tuned  densenet121   224     0.6389   

                                              accuracy  precision  recall  \
experiment   unit           mode                                            
densenet_hn2 image          default_0.5         0.9841     0.9956  0.9830   
                            validation_tuned    0.9776     0.9976  0.9722   
             filename_group default_0.5         0.9804     0.9930  0.9776   
                            validation_tuned    0.9771     0.9954  0.9702   

                                              specificity      f1  bal_acc  \
experiment   unit           mode                                             
densenet_hn2 image          default_0.5            0.9874  0.9892   0.9852   
                            validation_tuned       0.9933  0.9847   0.9828   
             filename_group default_0.5            0.9861  0.9852   0.9818   
                            validation_tuned       0.9910  0.9826   0.9806   

                                                 auc  pr_auc  
experiment   unit           mode                              
densenet_hn2 image          default_0.5       0.9985  0.9995  
                            validation_tuned  0.9985  0.9995  
             filename_group default_0.5       0.9979  0.9990  
                            validation_tuned  0.9979  0.9990

## 3.3. Khóa ngưỡng

In [20]:
if len(OOF) != 1:
    raise AssertionError("Stage A1 phải có đúng một cấu hình OOF.")
BEST = next(iter(OOF))
BEST_SPEC = next(spec for spec in EXPERIMENTS if spec["name"] == BEST)
IMAGE_THRESHOLD = OOF[BEST]["image_threshold"]
GROUP_THRESHOLD = OOF[BEST]["group_threshold"]

print(f"Cấu hình Stage A1 duy nhất; ngưỡng đã khóa bằng {DEV_LABEL}: {BEST}")
print(f"  group AUC          : {OOF[BEST]['group_default']['auc']:.4f}")
print(f"  threshold mức ảnh  : {IMAGE_THRESHOLD:.4f}")
print(f"  threshold mức group: {GROUP_THRESHOLD:.4f}")
print(f"  objective          : {THRESHOLD_OBJECTIVE}",
      f"(sensitivity >= {TARGET_SENSITIVITY:.0%})"
      if THRESHOLD_OBJECTIVE == "sensitivity" else "")
assert "test_probs" not in RUNS[0], "Test đã bị đọc trước khi khóa cấu hình"
# Một nguồn sự thật cho khâu tiền xử lý của cấu hình đã khóa. Mọi thứ phía sau
# (test, Grad-CAM, che vùng, mục 4.5) phải dùng đúng biến này.
BEST_RESIZE = BEST_SPEC.get("resize", RESIZE_MODE)
XAI_RESIZE = BEST_RESIZE
XAI_CACHE = IMAGE_CACHES[BEST_RESIZE]
print(f"  tiền xử lý         : {BEST_RESIZE}")

Cấu hình Stage A1 duy nhất; ngưỡng đã khóa bằng OOF validation: densenet_hn2
  group AUC          : 0.9979
  threshold mức ảnh  : 0.7020
  threshold mức group: 0.6389
  objective          : sensitivity (sensitivity >= 97%)
  tiền xử lý         : stretch


## 3.4. Known benchmark

Chỉ đọc sau khi ngưỡng đã khóa bằng OOF.

In [21]:
def show_confusion(title, matrix):
    (tn, fp), (fn, tp) = matrix
    sensitivity, specificity = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    precision = tp / max(tp + fp, 1)
    print(f"\n{title}  (n={tn + fp + fn + tp})")
    print(f"  TN {tn:>4}   FP {fp:>4}")
    print(f"  FN {fn:>4}   TP {tp:>4}")
    print(f"  bỏ sót {fn}/{fn + tp} ca viêm phổi  → độ nhạy {sensitivity:.1%}")
    print(f"  báo nhầm {fp}/{tn + fp} ca bình thường → độ đặc hiệu {specificity:.1%}")
    print(f"  precision {precision:.1%}  |  balanced accuracy "
          f"{(sensitivity + specificity) / 2:.1%}")


def load_model_from_run(run):
    model = build_model(run["arch"], pretrained=False)
    try:
        state = torch.load(run["checkpoint"], map_location="cpu", weights_only=True)
    except TypeError:  # PyTorch cũ
        state = torch.load(run["checkpoint"], map_location="cpu")
    model.load_state_dict(state)
    return model.eval()


best_runs = sorted((r for r in RUNS if r["experiment"] == BEST),
                   key=lambda r: r["fold"])
test_rows = FOLDS[0][FOLDS[0]["split"] == "test"].reset_index(drop=True)

# Ảnh test phải qua đúng khâu tiền xử lý mà mô hình đã được train. Dùng mặc
# định letterbox ở đây sẽ chấm một mô hình stretch trên phân phối đầu vào khác
# hẳn lúc train, và mọi chỉ số phía dưới đều sai mà không có dấu hiệu gì.
assert {run["resize"] for run in best_runs} == {BEST_RESIZE}, \
    "Các fold của BEST không cùng một chế độ resize."
test_loader = make_loader(FOLDS[0], "test", seed=SEED, mode=BEST_RESIZE)
print(f"Tiền xử lý dùng cho known benchmark test: {BEST_RESIZE}\n")

test_probabilities, test_labels = [], None
for run in best_runs:
    model = load_model_from_run(run)
    labels, probs = predict(model, test_loader, run["size"])
    if test_labels is None:
        test_labels = labels
    else:
        assert np.array_equal(test_labels, labels)
    test_probabilities.append(probs)
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()

test_probs = np.mean(test_probabilities, axis=0)
assert np.array_equal(test_labels, test_rows["class_id"].to_numpy())
test_groups = test_rows["group_id"].to_numpy()
group_labels, group_probs = to_group_level(test_groups, test_labels, test_probs)

FINAL = {
    "image_default": metrics_at(test_labels, test_probs, 0.5),
    "image_tuned": metrics_at(test_labels, test_probs, IMAGE_THRESHOLD),
    "group_default": metrics_at(group_labels, group_probs, 0.5),
    "group_tuned": metrics_at(group_labels, group_probs, GROUP_THRESHOLD),
    "test_labels": test_labels, "test_probs": test_probs,
    "group_labels": group_labels, "group_probs": group_probs,
}

final_rows = []
for unit, default_key, tuned_key in (
        ("image", "image_default", "image_tuned"),
        ("filename_group", "group_default", "group_tuned")):
    for mode, key in (("default_0.5", default_key), ("validation_tuned", tuned_key)):
        block = FINAL[key]
        final_rows.append({"experiment": BEST,
                           "arch": BEST_SPEC["arch"],
                           "resize": BEST_RESIZE,
                           "aug": BEST_SPEC["aug"],
                           "balancing": BEST_SPEC["balancing"],
                           "evaluation": "known_benchmark_not_final",
                           "unit": unit, "mode": mode,
                           "threshold": block["threshold"],
                           **{k: block[k] for k in METRIC_COLS},
                           "confusion_matrix": json.dumps(block["confusion_matrix"])})
final_results = pd.DataFrame(final_rows)
final_results.to_csv(WORK_DIR / "results_known_benchmark_test.csv", index=False)
benchmark_images = test_rows.assign(
    p_pneumonia=test_probs,
    pred=(test_probs >= IMAGE_THRESHOLD).astype(int),
    experiment=BEST, resize=BEST_RESIZE)
benchmark_images.to_csv(
    WORK_DIR / f"predictions_known_benchmark_{BEST}_images.csv", index=False)
benchmark_images.to_csv(
    WORK_DIR / "predictions_known_benchmark_test_images.csv", index=False)

best_group_predictions = (pd.DataFrame({
    "group_id": test_groups, "label": test_labels, "p_pneumonia": test_probs,
}).groupby("group_id", as_index=False)
  .agg(label=("label", "first"), p_pneumonia=("p_pneumonia", "mean")))
best_group_predictions["pred"] = (
    best_group_predictions["p_pneumonia"] >= GROUP_THRESHOLD).astype(int)
best_group_predictions["experiment"] = BEST
best_group_predictions["resize"] = BEST_RESIZE
best_group_predictions.to_csv(
    WORK_DIR / f"predictions_known_benchmark_{BEST}_groups.csv", index=False)
best_group_predictions.to_csv(
    WORK_DIR / "predictions_known_benchmark_test_groups.csv", index=False)

print(f"KNOWN BENCHMARK TEST — cấu hình đã khóa: {BEST}\n")
display(final_results.set_index(["unit", "mode"]).round(4))
show_confusion(f"{BEST} — TEST image @ {IMAGE_THRESHOLD:.3f}",
               FINAL["image_tuned"]["confusion_matrix"])
show_confusion(f"{BEST} — TEST filename-group @ {GROUP_THRESHOLD:.3f}",
               FINAL["group_tuned"]["confusion_matrix"])

# Loader BEST không còn dùng; giải phóng persistent workers trước mục 3.5.
del test_loader
gc.collect()

Tiền xử lý dùng cho known benchmark test: stretch

KNOWN BENCHMARK TEST — cấu hình đã khóa: densenet_hn2



experiment         arch   resize   aug  \
unit           mode                                                         
image          default_0.5       densenet_hn2  densenet121  stretch  manh   
               validation_tuned  densenet_hn2  densenet121  stretch  manh   
filename_group default_0.5       densenet_hn2  densenet121  stretch  manh   
               validation_tuned  densenet_hn2  densenet121  stretch  manh   

                                balancing                 evaluation  \
unit           mode                                                    
image          default_0.5       weighted  known_benchmark_not_final   
               validation_tuned  weighted  known_benchmark_not_final   
filename_group default_0.5       weighted  known_benchmark_not_final   
               validation_tuned  weighted  known_benchmark_not_final   

                                 threshold  accuracy  precision  recall  \
unit           mode                                                       
image          default_0.5          0.5000    0.9087     0.8742  0.9974   
               validation_tuned     0.7020    0.9343     0.9087  0.9949   
filename_group default_0.5          0.5000    0.8738     0.7922  0.9951   
               validation_tuned     0.6389    0.8949     0.8211  0.9951   

                                 specificity      f1  bal_acc     auc  pr_auc  \
unit           mode                                                             
image          default_0.5            0.7607  0.9317   0.8791  0.9846  0.9884   
               validation_tuned       0.8333  0.9498   0.9141  0.9846  0.9884   
filename_group default_0.5            0.7644  0.8821   0.8798  0.9801  0.9705   
               validation_tuned       0.8044  0.8998   0.8998  0.9801  0.9705   

                                      confusion_matrix  
unit           mode                                     
image          default_0.5       [[178, 56], [1, 389]]  
               validation_tuned  [[195, 39], [2, 388]]  
filename_group default_0.5       [[172, 53], [1, 202]]  
               validation_tuned  [[181, 44], [1, 202]]


densenet_hn2 — TEST image @ 0.702  (n=624)
  TN  195   FP   39
  FN    2   TP  388
  bỏ sót 2/390 ca viêm phổi  → độ nhạy 99.5%
  báo nhầm 39/234 ca bình thường → độ đặc hiệu 83.3%
  precision 90.9%  |  balanced accuracy 91.4%

densenet_hn2 — TEST filename-group @ 0.639  (n=428)
  TN  181   FP   44
  FN    1   TP  202
  bỏ sót 1/203 ca viêm phổi  → độ nhạy 99.5%
  báo nhầm 44/225 ca bình thường → độ đặc hiệu 80.4%
  precision 82.1%  |  balanced accuracy 90.0%


0

## 3.5. So với DenseNet v5 và ensemble

Điều kiện giữ đã thống nhất trước khi chạy: độ nhạy ≥0,97, và độ đặc hiệu ≥0,82
hoặc tăng ≥0,02 so với DenseNet, và giảm ròng ít nhất 4 ca báo nhầm.

In [22]:
group_row = FINAL["group_tuned"]
predictions = (FINAL["group_probs"] >= GROUP_THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(FINAL["group_labels"], predictions,
                                  labels=[0, 1]).ravel()

display(pd.DataFrame([
    {"model": "DenseNet121 v5", **BASELINE_DENSENET},
    {"model": "ensemble R+D", "specificity": BASELINE_ENSEMBLE["specificity"],
     "fp": BASELINE_ENSEMBLE["fp"], "sensitivity": 0.9951},
    {"model": "DenseNet HN2", "auc": group_row["auc"],
     "sensitivity": group_row["recall"],
     "specificity": group_row["specificity"],
     "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
]).round(4))

delta = group_row["specificity"] - BASELINE_DENSENET["specificity"]
avoided = BASELINE_DENSENET["fp"] - int(fp)
print(f"\nso với DenseNet v5: Δđặc hiệu {delta:+.4f}, tránh {avoided:+d} ca báo nhầm")
print(f"độ nhạy {group_row['recall']:.4f}")

keep = (group_row["recall"] >= 0.97
        and (group_row["specificity"] >= 0.82 or delta >= 0.02)
        and avoided >= 4)
print(f"\n=> {'GIỮ' if keep else 'KHÔNG GIỮ'} theo tiêu chí đã đặt trước.")
if RUN_MODE == "smoke":
    print("   (smoke run — chỉ kiểm tra pipeline)")

,model,auc,sensitivity,specificity,tn,fp,fn,tp
0,DenseNet121 v5,0.9801,0.9951,0.8044,181.0,44,1.0,202.0
1,ensemble R+D,NaN,0.9951,0.8222,NaN,40,NaN,NaN
2,DenseNet HN2,0.9801,0.9951,0.8044,181.0,44,1.0,202.0



so với DenseNet v5: Δđặc hiệu +0.0000, tránh +0 ca báo nhầm
độ nhạy 0.9951

=> KHÔNG GIỮ theo tiêu chí đã đặt trước.


# 4. Bước tiếp theo

Chưa triển khai ở đây, theo thứ tự đã thống nhất:

1. ensemble E1/E2/E3 từ prediction đã lưu, không cần train;
2. DeiT-Small dưới cùng pipeline;
3. ensemble cuối cùng.

Multiplier 1,5 chỉ chạy nếu điều kiện an toàn phía OOF được kích hoạt, không
phải vì benchmark cho kết quả không như ý.